This notebook is to combine data from several years to facilitate the model training

In [1]:
import pandas as pd                            
import os
import re
from collections import defaultdict
import os

In [2]:
def all_files_in(data_path):
    """
    Collects the paths of all files in the specified directory and its subdirectories.

    Args:
        data_path (str): The path of the directory to search for files.

    Returns:
        list: A list containing the full paths of all files found within the directory.
        
    Raises:
        ValueError: If the specified path is not a directory.
    """
    if not os.path.isdir(data_path):
        raise ValueError(f"The specified path '{data_path}' is not a valid directory.")
    
    all_files = []
    for dir_path, _, file_names in os.walk(data_path):
        for file_name in file_names:
            all_files.append(os.path.join(dir_path, file_name))
    
    return all_files

In [29]:
def categorize_files(file_list):
    categorized_files = defaultdict(list)
    
    # Define the regex patterns for the different categories
    patterns = {
        'lt1': r"data-storm-zambia-obj-to-map-(\d{4})-lt1.csv$",
        'lt0': r"data-storm-zambia-obj-to-map-(\d{4})-lt0.csv$",
        'lt3': r"data-storm-zambia-obj-to-map-(\d{4})-lt3.csv$",
        'input_t0': r"data-storm-zambia-obj-to-map-(\d{4})-input-t0.csv$"
    }

    # Iterate over each file in the input list
    for file in file_list:
        filename = os.path.basename(file)                # Extract the filename from the absolute path
        for category, pattern in patterns.items():
            # Match the filename against the pattern
            if re.match(pattern, filename):
                categorized_files[category].append(file)
    
    # Return lists for each category
    return categorized_files['input_t0'], categorized_files['lt0'], categorized_files['lt1'], categorized_files['lt3']

In [30]:
def merge_csv_from_list(csv_files, output_filename, keep_header=True):
    """
    Merges a list of CSV files, keeps the order, and handles headers based on user input.
    The merged file will be saved directly.

    Args:
    - csv_files: List of CSV file paths to merge
    - keep_header: Whether to keep the header (True to keep the header, False to ignore it)
    - output_filename: The filename to save the merged CSV (default is 'merged_output.csv')
    """
    merged_df = pd.DataFrame()

    for i, file in enumerate(csv_files):
        # Read each CSV file
        if i == 0 or keep_header:  # Keep header for the first file or if keep_header is True
            df = pd.read_csv(file)
        else:  # Skip header for subsequent files if keep_header is False
            df = pd.read_csv(file, header=None)
            # Skip the header row (if any) by renaming columns to match the first file
            df.columns = merged_df.columns

        # Merge the data
        merged_df = pd.concat([merged_df, df], ignore_index=True)

    # Save the merged DataFrame directly to a CSV
    merged_df.to_csv(output_filename, index=False)

In [31]:
def merge_binary_csv(csv_files, output_filename):
    """
    Merges a list of CSV files containing flattened binary matrices, without adding headers.
    The merged file will be saved directly without any headers.

    Args:
    - csv_files: List of CSV file paths to merge
    - output_filename: The filename to save the merged CSV (default is 'merged_binary_matrix.csv')
    """
    merged_df = pd.DataFrame()

    for i, file in enumerate(csv_files):
        # Read the CSV without headers (since it's a flattened binary matrix)
        df = pd.read_csv(file, header=None)

        # Concatenate the current file's data to the merged dataframe
        merged_df = pd.concat([merged_df, df], ignore_index=True)

    # Save the merged dataframe directly to a CSV, without index or header
    merged_df.to_csv(output_filename, index=False, header=False)

In [32]:
data_type = "test"
parent_folder = f"/home/mmmhr/test-zambia/obj-to-map/{data_type}"

all_files = all_files_in(parent_folder)
files_input_t0, files_map_t0, files_map_t1, files_map_t3 = categorize_files(all_files)
files_input_t0.sort()
files_map_t0.sort()
files_map_t1.sort()
files_map_t3.sort()

In [33]:
merge_csv_from_list(files_input_t0, f"../obj-to-map/results-merged/{data_type}-data-zambia-input-ob-t0.csv")
merge_binary_csv(files_map_t0, f"../obj-to-map/results-merged/{data_type}-data-zambia-map-t0.csv")
merge_binary_csv(files_map_t1, f"../obj-to-map/results-merged/{data_type}-data-zambia-map-t1.csv")
merge_binary_csv(files_map_t3, f"../obj-to-map/results-merged/{data_type}-data-zambia-map-t3.csv")